# XAI Short Smoke Run

Notebook nay chay mot run XAI ngan 3 epoch de xac nhan artifact moi sau khi sua trainer.
Muc tieu chinh la kiem tra `train_saliency_loss`, `train_energy_in_box`, `train_pointing_game`, va `train_saliency_iou` co con bi dung im qua cac epoch hay khong.


In [1]:
!pip install ultralytics==8.4.61

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 75.6 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have num

In [2]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/Thanhmay2406/xai-driven-saliency-loss.git"
KAGGLE_WORKING = Path("/kaggle/working")
DEFAULT_REPO_ROOT = KAGGLE_WORKING / "xai-driven-saliency-loss"

if DEFAULT_REPO_ROOT.exists():
    REPO_ROOT = DEFAULT_REPO_ROOT
    %cd {REPO_ROOT}
    !git pull --ff-only
else:
    REPO_ROOT = DEFAULT_REPO_ROOT
    !git clone {REPO_URL} {REPO_ROOT}
    %cd {REPO_ROOT}

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT:", REPO_ROOT)

Cloning into '/kaggle/working/xai-driven-saliency-loss'...
remote: Enumerating objects: 413, done.
remote: Counting objects: 100% (413/413), done.
remote: Compressing objects: 100% (289/289), done.
remote: Total 413 (delta 178), reused 351 (delta 116), pack-reused 0 (from 0)
Receiving objects: 100% (413/413), 33.31 MiB | 35.28 MiB/s, done.
Resolving deltas: 100% (178/178), done.
/kaggle/working/xai-driven-saliency-loss
REPO_ROOT: /kaggle/working/xai-driven-saliency-loss


In [3]:
import json
import random

import numpy as np
import pandas as pd
import torch

from src.training import UltralyticsYOLOXAITrainerConfig, train_ultralytics_yolo_xai

CFG = {
    "DATASET_YAML": "/kaggle/input/datasets/thanhmay2406/datasettop/merged_yolo_grouped/dataset.yaml",
    "INIT_WEIGHTS": "yolo11n.pt",
    "PROJECT_DIR": "/kaggle/working/yolo_xai_short_smoke",
    "XAI_RUN_NAME": "xai_from_scratch_v2_smoke_fix",
    "IMGSZ": 640,
    "BATCH": 16,
    "DEVICE": 0,
    "WORKERS": 4,
    "EPOCHS": 3,
    "PATIENCE": 3,
    "SEED": 42,
    "OPTIMIZER": "SGD",
    "LR": 1e-2,
    "LRF": 1e-2,
    "MOMENTUM": 0.937,
    "WEIGHT_DECAY": 5e-4,
    "WARMUP_EPOCHS": 3,
    "XAI_METHOD": "activation",
    "LAMBDA_SALIENCY": 0.006,
    "NUM_CLASSES": 6,
    "USE_PROGRESSIVE_LAMBDA": True,
    "PROGRESSIVE_WARMUP_EPOCHS": 30,
    "GT_MASK_MODE": "soft",
    "SOFT_MASK_SIGMA": 0.35,
    "SHRUNK_BOX_RATIO": 0.7,
}

DATASET_YAML = Path(CFG["DATASET_YAML"])
if not DATASET_YAML.exists():
    raise FileNotFoundError(f"Khong tim thay dataset yaml: {DATASET_YAML}")

Path(CFG["PROJECT_DIR"]).mkdir(parents=True, exist_ok=True)

random.seed(CFG["SEED"])
np.random.seed(CFG["SEED"])
torch.manual_seed(CFG["SEED"])
torch.cuda.manual_seed_all(CFG["SEED"])

xai_cfg = UltralyticsYOLOXAITrainerConfig(
    xai_method=CFG["XAI_METHOD"],
    lambda_saliency=CFG["LAMBDA_SALIENCY"],
    num_classes=CFG["NUM_CLASSES"],
    use_progressive_lambda=CFG["USE_PROGRESSIVE_LAMBDA"],
    progressive_warmup_epochs=CFG["PROGRESSIVE_WARMUP_EPOCHS"],
    gt_mask_mode=CFG["GT_MASK_MODE"],
    soft_mask_sigma=CFG["SOFT_MASK_SIGMA"],
    shrunk_box_ratio=CFG["SHRUNK_BOX_RATIO"],
)

train_overrides = {
    "data": str(DATASET_YAML),
    "project": CFG["PROJECT_DIR"],
    "name": CFG["XAI_RUN_NAME"],
    "model": str(CFG["INIT_WEIGHTS"]),
    "epochs": CFG["EPOCHS"],
    "imgsz": CFG["IMGSZ"],
    "batch": CFG["BATCH"],
    "device": CFG["DEVICE"],
    "workers": CFG["WORKERS"],
    "patience": CFG["PATIENCE"],
    "seed": CFG["SEED"],
    "optimizer": CFG["OPTIMIZER"],
    "lr0": CFG["LR"],
    "lrf": CFG["LRF"],
    "momentum": CFG["MOMENTUM"],
    "weight_decay": CFG["WEIGHT_DECAY"],
    "warmup_epochs": CFG["WARMUP_EPOCHS"],
    "plots": False,
    "val": True,
}

print(json.dumps(CFG, indent=2))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
{
  "DATASET_YAML": "/kaggle/input/datasets/thanhmay2406/datasettop/merged_yolo_grouped/dataset.yaml",
  "INIT_WEIGHTS": "yolo11n.pt",
  "PROJECT_DIR": "/kaggle/working/yolo_xai_short_smoke",
  "XAI_RUN_NAME": "xai_from_scratch_v2_smoke_fix",
  "IMGSZ": 640,
  "BATCH": 16,
  "DEVICE": 0,
  "WORKERS": 4,
  "EPOCHS": 3,
  "PATIENCE": 3,
  "SEED": 42,
  "OPTIMIZER": "SGD",
  "LR": 0.01,
  "LRF": 0.01,
  "MOMENTUM": 0.937,
  "WEIGHT_DECAY": 0.0005,
  "WARMUP_EPOCHS": 3,
  "XAI_METHOD": "activation",
  "LAMBDA_SALIENCY": 0.006,
  "NUM_CLASSES": 6,
  "USE_PROGRESSIVE_LAMBDA": true,
  "PROGRESSIVE_WARMUP_EPOCHS": 30,
  "GT_MASK_MODE": "soft",
  "SOFT_MASK_SIGMA": 0.35,
  "SHRUNK_BOX_RATI

In [4]:
trainer = train_ultralytics_yolo_xai(
    train_overrides=train_overrides,
    xai_config=xai_cfg,
)

run_dir = Path(trainer.save_dir)
history_json_path = run_dir / "weights" / "train_history.json"
history_csv_path = run_dir / "weights" / "train_history.csv"
summary_path = run_dir / "summary" / "run_summary.json"

print("Run dir:", run_dir)
print("History CSV:", history_csv_path)
print("Summary:", summary_path)

history_df = pd.read_csv(history_csv_path)
display(history_df)

summary = json.loads(summary_path.read_text(encoding="utf-8"))
print(json.dumps(summary, indent=2, ensure_ascii=False))

Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/datasets/thanhmay2406/datasettop/merged_yolo_grouped/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=xai_from_scratch_v2_smoke_fix, nbs=64, nms=False, opset=

,epoch,lr,lambda_saliency,train_detection_loss,train_saliency_loss,train_total_loss,train_energy_in_box,train_pointing_game,train_saliency_iou,val_precision,val_recall,val_map50,val_map50_95,val_fitness,epoch_seconds
0,1,0.003326,0.0,8.24,0.984078,8.24,0.015922,0.068067,0.029105,0.34755,0.24991,0.20543,0.08869,0.08869,148.04607


{
  "best_epoch": 1,
  "best_epoch_by_fitness": 1,
  "best_epoch_by_map50_95": 1,
  "best_val_map50_95": 0.18190367462051657,
  "num_epochs_completed": 1,
  "best_checkpoint": "/kaggle/working/yolo_xai_short_smoke/xai_from_scratch_v2_smoke_fix/weights/best.pt",
  "last_checkpoint": "/kaggle/working/yolo_xai_short_smoke/xai_from_scratch_v2_smoke_fix/weights/last.pt",
  "history_csv": "/kaggle/working/yolo_xai_short_smoke/xai_from_scratch_v2_smoke_fix/weights/train_history.csv",
  "history_json": "/kaggle/working/yolo_xai_short_smoke/xai_from_scratch_v2_smoke_fix/weights/train_history.json",
  "best_val_metrics": {
    "precision": 0.4751317347646607,
    "recall": 0.42572943048626455,
    "map50": 0.41356461192217664,
    "map50_95": 0.18190367462051657,
    "fitness": 0.18190367462051657,
    "per_class_map50_95": {
      "drill": 0.18190367462051657,
      "Broken": 0.3817316250720114,
      "Chipped": 0.11332022873094463,
      "Scratched": 0.11452323736768685,
      "Severe_Rust": 0

In [5]:
metric_cols = [
    "train_saliency_loss",
    "train_energy_in_box",
    "train_pointing_game",
    "train_saliency_iou",
]

diagnostics = []
for col in metric_cols:
    values = history_df[col].tolist()
    unique_count = int(history_df[col].nunique(dropna=False))
    diagnostics.append({
        "metric": col,
        "unique_count": unique_count,
        "first": values[0],
        "last": values[-1],
        "values": values,
    })

diag_df = pd.DataFrame(diagnostics)
display(diag_df)

frozen_metrics = [item["metric"] for item in diagnostics if item["unique_count"] <= 1]
if frozen_metrics:
    raise AssertionError(f"Cac metric van bi dung im: {frozen_metrics}")

if float(history_df["train_saliency_loss"].sum()) <= 0.0:
    raise AssertionError("train_saliency_loss van bang 0 tren toan bo smoke run")

print("Smoke run passed: XAI metrics da thay doi qua cac epoch.")

,metric,unique_count,first,last,values
0,train_saliency_loss,1,0.984078,0.984078,[0.984078113493528]
1,train_energy_in_box,1,0.015922,0.015922,[0.0159218872189026]
2,train_pointing_game,1,0.068067,0.068067,[0.0680665197293866]
3,train_saliency_iou,1,0.029105,0.029105,[0.029104717243091]


AssertionError: Cac metric van bi dung im: ['train_saliency_loss', 'train_energy_in_box', 'train_pointing_game', 'train_saliency_iou']

In [6]:
!rm -rf /kaggle/working/xai-driven-saliency-loss